In [1]:

# ================================================================
# EPSS MONTHLY TIME SERIES FORECASTING
# PyTorch | Three-layer LSTM | 2024-2025 historical data
# Forecast horizon: January-December 2026
# ================================================================

# 1. IMPORT LIBRARIES

import copy
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
import os

from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
# Folder containing the two historical CSV files
from google.colab import files

# choose the file on your computer to upload it then
csv_files = files.upload()

# Check that exactly two files were found
if len(csv_files) != 2:
    raise ValueError(
        f"Expected 2 CSV files, but found {len(csv_files)}."
    )

# Read and combine both files
historical_df = pd.concat(
    [pd.read_csv(file) for file in csv_files],
    ignore_index=True
)

# Convert month column to datetime
historical_df["month"] = pd.to_datetime(historical_df["month"])

# Sort chronologically by CVE and month
historical_df = historical_df.sort_values(
    ["cve_id", "month"]
).reset_index(drop=True)

print(f"Combined {len(csv_files)} files.")
print(f"Total records: {len(historical_df)}")
print(historical_df.head())

ValueError: Expected 2 CSV files, but found 0.

In [8]:
historical_df.columns

Index(['month', 'cve_id', 'epss', 'in_kev', 'exploitdb', 'metasploit',
       'nuclei', 'poc_github'],
      dtype='object')

In [11]:
# ================================================================
# 2. CUSTOMIZABLE SETTINGS
# ================================================================

# FILE LOCATIONS
OUTPUT_FILE = "EPSS_2026_Predictions.csv"
MODEL_FILE = "EPSS_LSTM_Model.pth"
SCALER_FILE = "EPSS_Scaler.pkl"

# COLUMN NAMES
ID_COLUMN = "cve_id"
DATE_COLUMN = "month"
TARGET_COLUMN = "epss"

# HISTORICAL PERIOD
START_DATE = "2024-01-01"
END_DATE = "2025-12-01"

# FORECAST PERIOD
FORECAST_START = "2026-01-01"
FORECAST_END = "2026-12-01"

# SEQUENCE SETTINGS
LOOKBACK = 6
# Number of previous months supplied to the model.
# Try 3, 6, or 12 to compare performance.

# LSTM SETTINGS
HIDDEN_SIZE = 64
NUM_LSTM_LAYERS = 3

DROPOUT_RATE = 0.20
# Higher dropout adds more regularization.
# Typical range: 0.1 - 0.5.

# FULLY CONNECTED PREDICTION HEAD
DENSE_SIZE_1 = 64
DENSE_SIZE_2 = 32

# TRAINING SETTINGS
LEARNING_RATE = 0.001
BATCH_SIZE = 128
EPOCHS = 10
PATIENCE = 2

# RANDOMNESS
RANDOM_SEED = 42

# COMPUTING DEVICE
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Using device:", DEVICE)

Using device: cpu


In [12]:
# ================================================================
# 3. IMPORT AND PREPARE HISTORICAL DATA
# ================================================================

df = historical_df

required_columns = [
    DATE_COLUMN,
    ID_COLUMN,
    TARGET_COLUMN
]

for column in required_columns:
    if column not in df.columns:
        raise ValueError(f"Missing required column: {column}")

df = df[required_columns].copy()

# Normalize dates to the first day of each month.
df[DATE_COLUMN] = (
    pd.to_datetime(df[DATE_COLUMN])
    .dt.to_period("M")
    .dt.to_timestamp()
)

df[ID_COLUMN] = df[ID_COLUMN].astype(str)

df[TARGET_COLUMN] = pd.to_numeric(
    df[TARGET_COLUMN], errors="coerce"
)

# Remove records without a CVE identifier or date.
df = df.dropna(subset=[ID_COLUMN, DATE_COLUMN])

# Check EPSS values.
valid_epss = df[TARGET_COLUMN].dropna()

if not valid_epss.between(0, 1).all():
    raise ValueError("EPSS values must be between 0 and 1.")

# Check duplicate monthly observations.
if df.duplicated([ID_COLUMN, DATE_COLUMN]).any():
    raise ValueError(
        "Duplicate CVE/month records detected. "
        "Resolve duplicates before training."
    )

# Restrict historical period.
df = df[
    (df[DATE_COLUMN] >= START_DATE) &
    (df[DATE_COLUMN] <= END_DATE)
].copy()

# Chronological ordering.
df = df.sort_values(
    [ID_COLUMN, DATE_COLUMN]
).reset_index(drop=True)

# Establish complete monthly calendar.
historical_months = pd.date_range(
    START_DATE,
    END_DATE,
    freq="MS"
)

forecast_months = pd.date_range(
    FORECAST_START,
    FORECAST_END,
    freq="MS"
)

print("Historical records:", len(df))
print("Unique CVEs:", df[ID_COLUMN].nunique())
print("Historical months:", len(historical_months))

Historical records: 6463734
Unique CVEs: 319691
Historical months: 24


In [13]:
# ================================================================
# 4. CREATE MONTHLY SEQUENCES FOR EACH CVE
# ================================================================

# Every CVE receives a complete monthly timeline.
#
# Missing monthly observations are NOT treated as zero.
#
# Instead:
# 1. Carry forward the last known EPSS value.
# 2. Create an observation indicator.
# 3. Use the indicator to tell the model whether the value
#    was actually observed or filled in.

# Use training-period median for initial missing values.
training_observations = df.loc[
    df[DATE_COLUMN] <= "2024-12-01",
    TARGET_COLUMN
].dropna()

GLOBAL_MEDIAN = training_observations.median()

if pd.isna(GLOBAL_MEDIAN):
    raise ValueError("No valid training EPSS observations.")

cve_sequences = {}

for cve, group in df.groupby(ID_COLUMN):

    monthly_values = (
        group.set_index(DATE_COLUMN)[TARGET_COLUMN]
        .reindex(historical_months)
    )

    # 1 = actual observation
    # 0 = missing observation

    observation_mask = (
        monthly_values.notna().astype(float).to_numpy()
    )

    # Carry last known observation forward.
    filled_values = monthly_values.ffill()

    # For CVEs without an earlier observation,
    # use the training-period median.
    filled_values = filled_values.fillna(GLOBAL_MEDIAN)

    cve_sequences[cve] = {
        "values": filled_values.to_numpy(dtype=float),
        "mask": observation_mask
    }

print("Monthly sequences created:", len(cve_sequences))

Monthly sequences created: 319691


In [14]:
# ================================================================
# 5. NORMALIZE HISTORICAL EPSS VALUES
# ================================================================

# Fit scaler only on training-period observations.
# Validation and test data must not influence this scaler.

scaler = StandardScaler()

scaler.fit(
    training_observations.to_numpy().reshape(-1, 1)
)

StandardScaler()

In [15]:
# ================================================================
# 6. CREATE TIME-BASED INPUT FEATURES
# ================================================================

def create_month_features(month):

    # Cyclical month representation.
    # Helps the model recognize annual seasonality.

    month_number = month.month

    month_sin = np.sin(
        2 * np.pi * month_number / 12
    )

    month_cos = np.cos(
        2 * np.pi * month_number / 12
    )

    return [month_sin, month_cos]


def create_sequence(values, mask, months, scaler):

    # Normalize historical EPSS values.
    normalized_values = scaler.transform(
        np.asarray(values).reshape(-1, 1)
    ).flatten()

    sequence = []

    for i in range(len(values)):

        time_features = create_month_features(months[i])

        sequence.append([
            normalized_values[i],
            mask[i],
            time_features[0],
            time_features[1]
        ])

    return np.array(sequence, dtype=np.float32)

In [16]:
# ================================================================
# 7. CREATE SLIDING-WINDOW TRAINING EXAMPLES
# ================================================================

def create_samples(
    sequences,
    months,
    scaler,
    target_start,
    target_end
):

    X = []
    y = []

    for cve, data in sequences.items():

        values = data["values"]
        mask = data["mask"]

        for target_index in range(LOOKBACK, len(months)):

            target_month = months[target_index]

            # Only use targets within the requested period.
            if not (target_start <= target_month <= target_end):
                continue

            # Do not train on missing target observations.
            if mask[target_index] == 0:
                continue

            # Historical inputs only.
            start_index = target_index - LOOKBACK

            historical_values = values[
                start_index:target_index
            ]

            historical_mask = mask[
                start_index:target_index
            ]

            historical_months = months[
                start_index:target_index
            ]

            sequence = create_sequence(
                historical_values,
                historical_mask,
                historical_months,
                scaler
            )

            X.append(sequence)

            # Target is the actual EPSS score for the
            # month immediately following the input window.
            y.append(values[target_index])

    return (
        np.array(X, dtype=np.float32),
        np.array(y, dtype=np.float32).reshape(-1, 1)
    )

In [19]:

# ================================================================
# 8. SPLIT TRAINING / VALIDATION / TEST DATA
# ================================================================

# Chronological split across the entire historical period:
# 70% Training, 15% Validation, 15% Testing

total_months = len(historical_months)

train_end_index = int(total_months * 0.70)
validation_end_index = int(total_months * 0.85)

# Identify the target months for each dataset
train_start = historical_months[0]
train_end = historical_months[train_end_index - 1]

validation_start = historical_months[train_end_index]
validation_end = historical_months[validation_end_index - 1]

test_start = historical_months[validation_end_index]
test_end = historical_months[-1]

print("CHRONOLOGICAL DATA SPLITS")
print(f"Training:   {train_start:%Y-%m} to {train_end:%Y-%m}")
print(f"Validation: {validation_start:%Y-%m} to {validation_end:%Y-%m}")
print(f"Testing:    {test_start:%Y-%m} to {test_end:%Y-%m}")


# Create training samples
X_train, y_train = create_samples(
    cve_sequences,
    historical_months,
    scaler,
    train_start,
    train_end
)
print("\nTraining data obtained")


# Create validation samples
X_validation, y_validation = create_samples(
    cve_sequences,
    historical_months,
    scaler,
    validation_start,
    validation_end
)
print("Validation data obtained")


# Create testing samples
X_test, y_test = create_samples(
    cve_sequences,
    historical_months,
    scaler,
    test_start,
    test_end
)
print("Testing data obtained")


# Display dataset sizes
print("\nDATASET SIZES")
print("Training:", len(X_train))
print("Validation:", len(X_validation))
print("Testing:", len(X_test))

# Confirm that the target-month splits do not overlap
assert train_end < validation_start
assert validation_end < test_start

if min(len(X_train), len(X_validation), len(X_test)) == 0:
    raise ValueError(
        "One of the datasets contains no training examples. "
        "Check historical coverage and missing EPSS values."
    )


CHRONOLOGICAL DATA SPLITS
Training:   2024-01 to 2025-04
Validation: 2025-05 to 2025-08
Testing:    2025-09 to 2025-12

Training data obtained
Validation data obtained
Testing data obtained

DATASET SIZES
Training: 2664390
Validation: 1144380
Testing: 1207286


In [20]:
# ================================================================
# 9. CONVERT DATA INTO PYTORCH TENSORS
# ================================================================

def create_loader(X, y, shuffle=False):

    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle
    )


train_loader = create_loader(
    X_train, y_train, shuffle=True
)

validation_loader = create_loader(
    X_validation, y_validation
)

test_loader = create_loader(
    X_test, y_test
)

In [21]:
# ================================================================
# 10. DEFINE THREE-LAYER LSTM MODEL
# ================================================================

class EPSSLSTM(nn.Module):

    def __init__(self):

        super(EPSSLSTM, self).__init__()

        # INPUT FEATURES:
        # 1. Normalized EPSS
        # 2. Observation indicator
        # 3. Month sine
        # 4. Month cosine

        INPUT_SIZE = 4

        # THREE RECURRENT LAYERS
        self.lstm = nn.LSTM(
            input_size=INPUT_SIZE,
            hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LSTM_LAYERS,
            batch_first=True,
            dropout=DROPOUT_RATE
        )

        # NORMALIZATION
        self.norm = nn.LayerNorm(HIDDEN_SIZE)

        # THREE-LAYER DENSE PREDICTION HEAD

        self.fc1 = nn.Linear(
            HIDDEN_SIZE, DENSE_SIZE_1
        )

        self.fc2 = nn.Linear(
            DENSE_SIZE_1, DENSE_SIZE_2
        )

        self.fc3 = nn.Linear(
            DENSE_SIZE_2, 1
        )

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(DROPOUT_RATE)

        # Constrain output to EPSS probability range.
        self.sigmoid = nn.Sigmoid()


    def forward(self, x):

        # Process historical sequence.
        lstm_output, _ = self.lstm(x)

        # Extract final timestep.
        x = lstm_output[:, -1, :]

        # Normalize activations.
        x = self.norm(x)

        # Dense layer 1.
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)

        # Dense layer 2.
        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)

        # Output layer.
        x = self.fc3(x)

        # EPSS range: 0 to 1.
        x = self.sigmoid(x)

        return x

In [22]:
# ================================================================
# 11. MODEL TRAINING FUNCTION
# ================================================================

def train_model(
    model,
    train_loader,
    validation_loader,
    epochs
):

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    best_validation_loss = float("inf")
    best_epoch = 0
    best_weights = None
    patience_counter = 0

    for epoch in range(epochs):

        # ---------------- TRAINING ----------------

        model.train()

        training_loss = 0.0

        for batch_X, batch_y in train_loader:

            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()

            predictions = model(batch_X)

            loss = criterion(
                predictions, batch_y
            )

            loss.backward()

            optimizer.step()

            training_loss += (
                loss.item() * len(batch_X)
            )

        training_loss /= len(train_loader.dataset)


        # ---------------- VALIDATION ----------------

        model.eval()

        validation_loss = 0.0

        with torch.no_grad():

            for batch_X, batch_y in validation_loader:

                batch_X = batch_X.to(DEVICE)
                batch_y = batch_y.to(DEVICE)

                predictions = model(batch_X)

                loss = criterion(
                    predictions, batch_y
                )

                validation_loss += (
                    loss.item() * len(batch_X)
                )

        validation_loss /= len(validation_loader.dataset)


        # ---------------- EARLY STOPPING ----------------

        if validation_loss < best_validation_loss:

            best_validation_loss = validation_loss

            best_epoch = epoch + 1

            best_weights = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1


        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Training Loss: {training_loss:.6f} | "
            f"Validation Loss: {validation_loss:.6f}"
        )

        if patience_counter >= PATIENCE:

            print("Early stopping activated.")
            break

    model.load_state_dict(best_weights)

    return model, best_epoch

In [25]:
# ================================================================
# 12. INITIALIZE AND TRAIN MODEL
# ================================================================

model = EPSSLSTM().to(DEVICE)

model, best_epoch = train_model(
    model,
    train_loader,
    validation_loader,
    EPOCHS
)

print("\nBest training epoch:", best_epoch)

Epoch 1/10 | Training Loss: 0.001526 | Validation Loss: 0.000300
Epoch 2/10 | Training Loss: 0.001250 | Validation Loss: 0.000266
Epoch 3/10 | Training Loss: 0.001224 | Validation Loss: 0.000331
Epoch 4/10 | Training Loss: 0.001213 | Validation Loss: 0.000205
Epoch 5/10 | Training Loss: 0.001208 | Validation Loss: 0.000251
Epoch 6/10 | Training Loss: 0.001199 | Validation Loss: 0.000268
Early stopping activated.

Best training epoch: 4


In [26]:
# ================================================================
# 13. TEST MODEL PERFORMANCE
# ================================================================

model.eval()

test_predictions = []
actual_values = []

with torch.no_grad():

    for batch_X, batch_y in test_loader:

        batch_X = batch_X.to(DEVICE)

        predictions = model(batch_X)

        test_predictions.extend(
            predictions.cpu().numpy().flatten()
        )

        actual_values.extend(
            batch_y.numpy().flatten()
        )


mae = mean_absolute_error(
    actual_values, test_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        actual_values, test_predictions
    )
)

print("\nTEST PERFORMANCE")
print("------------------------------")
print("MAE:", mae)
print("RMSE:", rmse)


TEST PERFORMANCE
------------------------------
MAE: 0.004338325333602366
RMSE: 0.016111854640924742


In [27]:
# ================================================================
# 14. RETRAIN FINAL MODEL USING ALL HISTORICAL DATA
# ================================================================

# After evaluation, refit using all available historical
# observations for the actual 2026 forecast.

final_scaler = StandardScaler()

all_observed_values = df[TARGET_COLUMN].dropna()

final_scaler.fit(
    all_observed_values.to_numpy().reshape(-1, 1)
)

# Generate final samples using all available historical months
X_final, y_final = create_samples(
    cve_sequences,
    historical_months,
    final_scaler,
    historical_months[0],
    historical_months[-1]
)

final_loader = create_loader(
    X_final,
    y_final,
    shuffle=True
)

final_model = EPSSLSTM().to(DEVICE)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=LEARNING_RATE
)

# Retrain using the best epoch count identified
# during validation-based early stopping.

for epoch in range(best_epoch):

    final_model.train()

    for batch_X, batch_y in final_loader:

        batch_X = batch_X.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad()

        predictions = final_model(batch_X)

        loss = criterion(predictions, batch_y)

        loss.backward()

        optimizer.step()

    print(
        f"Final training epoch: {epoch + 1}/{best_epoch}"
    )

final_model.eval()

# Save final model and scaler.
torch.save(
    final_model.state_dict(),
    MODEL_FILE
)

joblib.dump(
    final_scaler,
    SCALER_FILE
)

print("\nFinal model and scaler saved successfully.")


Final training epoch: 1/4
Final training epoch: 2/4
Final training epoch: 3/4
Final training epoch: 4/4

Final model and scaler saved successfully.


In [28]:
# ================================================================
# 15. GENERATE 2026 FORECASTS FOR EVERY HISTORICAL CVE
# ================================================================

forecast_results = []

# Set model to evaluation mode once before forecasting.
final_model.eval()

for cve, data in cve_sequences.items():

    # Start with the complete historical sequence.
    historical_values = list(data["values"])
    historical_mask = list(data["mask"])

    available_months = list(historical_months)

    # Ensure sufficient historical data is available.
    if len(historical_values) < LOOKBACK:
        print(f"Skipping {cve}: insufficient historical data.")
        continue

    # Forecast one month at a time.
    for forecast_month in forecast_months:

        # Select previous LOOKBACK months.
        recent_values = historical_values[-LOOKBACK:]
        recent_mask = historical_mask[-LOOKBACK:]
        recent_months = available_months[-LOOKBACK:]

        # Construct model input.
        sequence = create_sequence(
            recent_values,
            recent_mask,
            recent_months,
            final_scaler
        )

        input_tensor = torch.tensor(
            sequence,
            dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)

        # Generate prediction.
        with torch.no_grad():
            predicted_epss = final_model(
                input_tensor
            ).item()

        # Store forecast.
        forecast_results.append({

            ID_COLUMN: cve,

            "snapshot_month": forecast_month.strftime(
                "%Y-%m"
            ),

            TARGET_COLUMN: predicted_epss

        })

        # Append prediction for recursive forecasting.
        historical_values.append(predicted_epss)

        # Value is available, but model-generated.
        historical_mask.append(1.0)

        available_months.append(forecast_month)


# Convert forecasts into a DataFrame.
forecast_df = pd.DataFrame(forecast_results)

# Sort results by CVE and forecast month.
forecast_df = forecast_df.sort_values(
    [ID_COLUMN, "snapshot_month"]
).reset_index(drop=True)

print("\n2026 FORECAST SUMMARY")
print("Total predictions:", len(forecast_df))
print("Unique CVEs:", forecast_df[ID_COLUMN].nunique())
print("Forecast months:", forecast_df["snapshot_month"].nunique())

# Save predictions.
forecast_df.to_csv(
    "EPSS_2026_Predictions.csv",
    index=False
)


2026 FORECAST SUMMARY
Total predictions: 3836292
Unique CVEs: 319691
Forecast months: 12


In [29]:
# ================================================================
# 16. EXPORT FINAL PREDICTION FILE
# ================================================================

results = pd.DataFrame(forecast_results)

results = results[
    [ID_COLUMN, "snapshot_month", TARGET_COLUMN]
]

results.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nFORECASTING COMPLETED")
print("------------------------------")

print("Unique CVEs:", results[ID_COLUMN].nunique())

print("Forecast records:", len(results))

print("\nSample predictions:")

print(results.head(15))

print("\nSaved to:", OUTPUT_FILE)


FORECASTING COMPLETED
------------------------------
Unique CVEs: 319691
Forecast records: 3836292

Sample predictions:
           cve_id snapshot_month      epss
0   CVE-1999-0001        2026-01  0.006959
1   CVE-1999-0001        2026-02  0.005007
2   CVE-1999-0001        2026-03  0.026475
3   CVE-1999-0001        2026-04  0.023388
4   CVE-1999-0001        2026-05  0.017114
5   CVE-1999-0001        2026-06  0.011600
6   CVE-1999-0001        2026-07  0.007343
7   CVE-1999-0001        2026-08  0.004881
8   CVE-1999-0001        2026-09  0.003225
9   CVE-1999-0001        2026-10  0.001634
10  CVE-1999-0001        2026-11  0.000646
11  CVE-1999-0001        2026-12  0.000194
12  CVE-1999-0002        2026-01  0.095848
13  CVE-1999-0002        2026-02  0.085642
14  CVE-1999-0002        2026-03  0.104935

Saved to: EPSS_2026_Predictions.csv


In [ ]:
scorecard_2026 = files.upload()

In [48]:

# ================================================================
# 17. COMBINE SCOREBOARD AND MODEL PREDICTIONS
# ================================================================

# Load the existing scoreboard
scorecard_df = pd.read_csv("scoreboard_2026.csv")

# Use the model predictions generated in Part 15
predicted_files = forecast_df.copy()

ID_COLUMN = "cve_id"
DATE_COLUMN = "snapshot_month"
TARGET_COLUMN = "epss"

KEY_COLUMNS = [ID_COLUMN, DATE_COLUMN]

# ------------------------------------------------
# 17.2. VALIDATE REQUIRED COLUMNS
# ------------------------------------------------

required_columns = [
    ID_COLUMN,
    DATE_COLUMN,
    TARGET_COLUMN
]

for column in required_columns:

    if column not in scorecard_df.columns:
        raise ValueError(
            f"Missing column in scoreboard: {column}"
        )

    if column not in predicted_files.columns:
        raise ValueError(
            f"Missing column in predictions: {column}"
        )


# ------------------------------------------------
# 17.3. CLEAN AND STANDARDIZE DATA
# ------------------------------------------------

scorecard_df = scorecard_df[required_columns].copy()
predicted_files = predicted_files[required_columns].copy()

# Standardize CVE identifiers
for dataset in [scorecard_df, predicted_files]:

    dataset[ID_COLUMN] = (
        dataset[ID_COLUMN]
        .astype("string")
        .str.strip()
        .str.upper()
    )

# Standardize dates to the first day of each month
scorecard_df[DATE_COLUMN] = (
    pd.to_datetime(scorecard_df[DATE_COLUMN])
    .dt.to_period("M")
    .dt.to_timestamp()
)

predicted_files[DATE_COLUMN] = (
    pd.to_datetime(predicted_files[DATE_COLUMN])
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Ensure EPSS values are numeric
scorecard_df[TARGET_COLUMN] = pd.to_numeric(
    scorecard_df[TARGET_COLUMN],
    errors="coerce"
)

predicted_files[TARGET_COLUMN] = pd.to_numeric(
    predicted_files[TARGET_COLUMN],
    errors="coerce"
)


# ------------------------------------------------
# 17.4. CHECK CVE OVERLAP
# ------------------------------------------------

scorecard_ids = set(scorecard_df[ID_COLUMN].dropna())
prediction_ids = set(predicted_files[ID_COLUMN].dropna())

common_ids = scorecard_ids.intersection(prediction_ids)

print("\nCVE OVERLAP")
print("------------------------------")
print("Scoreboard CVEs:", len(scorecard_ids))
print("Prediction CVEs:", len(prediction_ids))
print("CVEs appearing in both:", len(common_ids))


# ------------------------------------------------
# 17.5. HANDLE DUPLICATE CVE/MONTH RECORDS
# ------------------------------------------------

# Preserve the first non-missing EPSS score for each key.
scorecard_df = (
    scorecard_df.groupby(KEY_COLUMNS, as_index=False)
    [TARGET_COLUMN]
    .first()
)

predicted_files = (
    predicted_files.groupby(KEY_COLUMNS, as_index=False)
    [TARGET_COLUMN]
    .first()
)


# ------------------------------------------------
# 17.6. COMBINE BOTH DATASETS
# ------------------------------------------------

combined_df = scorecard_df.merge(
    predicted_files,
    on=KEY_COLUMNS,
    how="outer",
    suffixes=("_scoreboard", "_predicted")
)

# Preserve existing scoreboard scores.
# Fill blank scoreboard scores with model predictions.
combined_df[TARGET_COLUMN] = (
    combined_df[f"{TARGET_COLUMN}_scoreboard"]
    .combine_first(
        combined_df[f"{TARGET_COLUMN}_predicted"]
    )
)

# Keep the required output columns
combined_df = combined_df[
    [ID_COLUMN, DATE_COLUMN, TARGET_COLUMN]
]


# ------------------------------------------------
# 17.7. SORT AND SAVE FINAL DATASET
# ------------------------------------------------

combined_df = combined_df.sort_values(
    [ID_COLUMN, DATE_COLUMN]
).reset_index(drop=True)

combined_df.to_csv(
    "EPSS_2026_Predictions_Full.csv",
    index=False
)


# ------------------------------------------------
# 17.8. FINAL DATASET SUMMARY
# ------------------------------------------------

print("\nFINAL EPSS DATASET")
print("--------------------------------")

print("Total records:", len(combined_df))
print("Unique CVEs:", combined_df[ID_COLUMN].nunique())

print(
    "Records with EPSS scores:",
    combined_df[TARGET_COLUMN].notna().sum()
)

print(
    "Records with missing EPSS:",
    combined_df[TARGET_COLUMN].isna().sum()
)

print(
    "Records filled by model:",
    (
        combined_df[TARGET_COLUMN].notna().sum()
        - scorecard_df[TARGET_COLUMN].notna().sum()
    )
)

print("\nMonthly record counts:")
print(
    combined_df.groupby(DATE_COLUMN)[ID_COLUMN].count()
)

print("\nFile saved: EPSS_2026_Predictions_Full.csv")


CVE OVERLAP
------------------------------
Scoreboard CVEs: 72670
Prediction CVEs: 319691
CVEs appearing in both: 0

FINAL EPSS DATASET
--------------------------------
Total records: 3908962
Unique CVEs: 392361
Records with EPSS scores: 3907531
Records with missing EPSS: 1431
Records filled by model: 3836292

Monthly record counts:
snapshot_month
2026-01-01    323986
2026-02-01    324294
2026-03-01    325918
2026-04-01    325492
2026-05-01    326613
2026-06-01    319691
2026-07-01    337369
2026-08-01    331933
2026-09-01    334593
2026-10-01    319691
2026-11-01    319691
2026-12-01    319691
Name: cve_id, dtype: int64

File saved: EPSS_2026_Predictions_Full.csv


In [6]:
# ------------------------------------------------
# 18. GET THE CVSS2 SCORES AND DESCRIPTIONS
# ------------------------------------------------

# Rereading the file (code is running on a new day)
sorted_df = pd.read_csv("EPSS_2026_Predictions_Full.csv")

sorted_df.sort_values(by='epss', ascending=False)

sorted_df.to_csv(
    "EPSS_2026_Predictions_Full_Sorted.csv",
    index=False
)

In [15]:
# Reading vulnerabilities file to get CVSS2 scores and description
ccvs_df = pd.read_csv("vulnerabilities.csv")

required_columns = [
    "cve_id",
    "cvss2_score",
    "cvss2_severity",
    "description",
    "cvss4_score",
    "cvss4_severity"
]

ccvs_df = ccvs_df[required_columns].copy()


In [16]:
# Merging the sorted values with the additional information
sorted_values_df = sorted_df.merge(
    ccvs_df,
    on="cve_id",
    how="left"
)

In [18]:
sorted_values_df.to_csv(
    "EPSS_CVSS24_Predictions.csv",
    index=False
)